# 👥 RetailPulse - Notebook 05A
# Customer Feature Engineering

## Objective

The objective of this notebook is to transform transaction-level retail data into customer-level features for machine learning applications.

The generated dataset will be used for:

- Customer Churn Prediction
- Recommendation System
- Customer Analytics Dashboard
- Power BI Dashboard

## Input

- data/processed/cleaned_retail_data.csv

## Output

- data/processed/customer_features.csv

In [3]:
# ============================================================
# Import Required Libraries
# ============================================================

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

print("Libraries Imported Successfully")

Libraries Imported Successfully


In [4]:
# ============================================================
# Project Paths
# ============================================================

BASE_DIR = Path.cwd().parent

DATA_DIR = BASE_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"

customer_output = PROCESSED_DIR / "customer_features.csv"

print("Processed Directory :", PROCESSED_DIR)

Processed Directory : c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\data\processed


In [5]:
# ============================================================
# List Processed Files
# ============================================================

import os

print("Files inside processed folder:\n")

for file in os.listdir(PROCESSED_DIR):
    print(file)

Files inside processed folder:

analysis_data.csv
cluster_validation.csv
customer_rfm.csv
customer_segments.csv
daily_sales.csv
daily_sales_features.csv
retail_cleaned.csv
test_sales.csv
train_sales.csv


In [6]:
# ============================================================
# Load Cleaned Retail Dataset
# ============================================================

retail_file = PROCESSED_DIR / "retail_cleaned.csv"

retail_df = pd.read_csv(
    retail_file,
    parse_dates=["InvoiceDate"]
)

print("=" * 60)
print("Retail Dataset Loaded Successfully")
print("=" * 60)

print(f"Shape : {retail_df.shape}")

display(retail_df.head())

Retail Dataset Loaded Successfully
Shape : (779425, 22)


,InvoiceID,StockCode,ProductDescription,Quantity,InvoiceDate,UnitPrice,CustomerID,Country,TotalAmount,InvoiceYear,InvoiceQuarter,InvoiceMonth,MonthName,InvoiceWeek,InvoiceDay,DayName,InvoiceHour,IsWeekend,CustomerCountry,InvoiceMonthYear,BasketSize,BasketValue
0,489434,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS,12,2009-12-01 07:45:00,6.95,13085.0,United Kingdom,83.4,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
1,489434,79323P,PINK CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
2,489434,79323W,WHITE CHERRY LIGHTS,12,2009-12-01 07:45:00,6.75,13085.0,United Kingdom,81.0,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
3,489434,22041,"RECORD FRAME 7"" SINGLE SIZE",48,2009-12-01 07:45:00,2.10,13085.0,United Kingdom,100.8,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3
4,489434,21232,STRAWBERRY CERAMIC TRINKET BOX,24,2009-12-01 07:45:00,1.25,13085.0,United Kingdom,30.0,2009,4,12,December,49,1,Tuesday,7,False,United Kingdom,2009-12,8,505.3


In [7]:
# ============================================================
# Dataset Validation
# ============================================================

print("Columns:")
print(retail_df.columns.tolist())

print("\nMissing Values")
display(retail_df.isnull().sum())

print("\nDate Range")
print(retail_df["InvoiceDate"].min())
print(retail_df["InvoiceDate"].max())

print("\nUnique Customers :", retail_df["CustomerID"].nunique())

Columns:
['InvoiceID', 'StockCode', 'ProductDescription', 'Quantity', 'InvoiceDate', 'UnitPrice', 'CustomerID', 'Country', 'TotalAmount', 'InvoiceYear', 'InvoiceQuarter', 'InvoiceMonth', 'MonthName', 'InvoiceWeek', 'InvoiceDay', 'DayName', 'InvoiceHour', 'IsWeekend', 'CustomerCountry', 'InvoiceMonthYear', 'BasketSize', 'BasketValue']

Missing Values


InvoiceID             0
StockCode             0
ProductDescription    0
Quantity              0
InvoiceDate           0
UnitPrice             0
CustomerID            0
Country               0
TotalAmount           0
InvoiceYear           0
InvoiceQuarter        0
InvoiceMonth          0
MonthName             0
InvoiceWeek           0
InvoiceDay            0
DayName               0
InvoiceHour           0
IsWeekend             0
CustomerCountry       0
InvoiceMonthYear      0
BasketSize            0
BasketValue           0
dtype: int64


Date Range
2009-12-01 07:45:00
2011-12-09 12:50:00

Unique Customers : 5878


In [8]:
# ============================================================
# Customer Feature Engineering
# ============================================================

snapshot_date = retail_df["InvoiceDate"].max() + pd.Timedelta(days=1)

customer_features = retail_df.groupby("CustomerID").agg(

    Recency=(
        "InvoiceDate",
        lambda x: (snapshot_date - x.max()).days
    ),

    Frequency=(
        "InvoiceID",
        "nunique"
    ),

    Monetary=(
        "TotalAmount",
        "sum"
    ),

    TotalQuantity=(
        "Quantity",
        "sum"
    ),

    AverageOrderValue=(
        "TotalAmount",
        "mean"
    ),

    UniqueProducts=(
        "StockCode",
        "nunique"
    ),

    UniqueCountries=(
        "CustomerCountry",
        "nunique"
    ),

    FirstPurchase=(
        "InvoiceDate",
        "min"
    ),

    LastPurchase=(
        "InvoiceDate",
        "max"
    )

).reset_index()

customer_features["CustomerLifetimeDays"] = (
    customer_features["LastPurchase"] -
    customer_features["FirstPurchase"]
).dt.days

print("Customer Feature Dataset Shape:", customer_features.shape)

display(customer_features.head())

Customer Feature Dataset Shape: (5878, 11)


,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AverageOrderValue,UniqueProducts,UniqueCountries,FirstPurchase,LastPurchase,CustomerLifetimeDays
0,12346.0,326,12,77556.46,74285,2281.072353,27,1,2009-12-14 08:34:00,2011-01-18 10:01:00,400
1,12347.0,2,8,4921.53,2967,22.169054,126,1,2010-10-31 14:20:00,2011-12-07 15:52:00,402
2,12348.0,75,5,2019.40,2714,39.596078,25,1,2010-09-27 14:59:00,2011-09-25 13:13:00,362
3,12349.0,19,4,4428.69,1624,25.306800,138,1,2010-04-29 13:20:00,2011-11-21 09:51:00,570
4,12350.0,310,1,334.40,197,19.670588,17,1,2011-02-02 16:01:00,2011-02-02 16:01:00,0


In [9]:
# ============================================================
# Additional Customer Features
# ============================================================

# Active months as a customer
customer_features["ActiveMonths"] = (
    (customer_features["LastPurchase"] - customer_features["FirstPurchase"]).dt.days / 30
).round(1)

customer_features["ActiveMonths"] = customer_features["ActiveMonths"].replace(0, 1)

# Revenue per month
customer_features["RevenuePerMonth"] = (
    customer_features["Monetary"] / customer_features["ActiveMonths"]
).round(2)

# Average quantity per order
customer_features["AvgQuantityPerOrder"] = (
    customer_features["TotalQuantity"] / customer_features["Frequency"]
).round(2)

display(customer_features.head())

,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AverageOrderValue,UniqueProducts,UniqueCountries,FirstPurchase,LastPurchase,CustomerLifetimeDays,ActiveMonths,RevenuePerMonth,AvgQuantityPerOrder
0,12346.0,326,12,77556.46,74285,2281.072353,27,1,2009-12-14 08:34:00,2011-01-18 10:01:00,400,13.3,5831.31,6190.42
1,12347.0,2,8,4921.53,2967,22.169054,126,1,2010-10-31 14:20:00,2011-12-07 15:52:00,402,13.4,367.28,370.88
2,12348.0,75,5,2019.40,2714,39.596078,25,1,2010-09-27 14:59:00,2011-09-25 13:13:00,362,12.1,166.89,542.80
3,12349.0,19,4,4428.69,1624,25.306800,138,1,2010-04-29 13:20:00,2011-11-21 09:51:00,570,19.0,233.09,406.00
4,12350.0,310,1,334.40,197,19.670588,17,1,2011-02-02 16:01:00,2011-02-02 16:01:00,0,1.0,334.40,197.00


In [10]:
# ============================================================
# Save Customer Features
# ============================================================

customer_output = PROCESSED_DIR / "customer_features.csv"

customer_features.to_csv(customer_output, index=False)

print("=" * 60)
print("Customer Features Saved Successfully")
print("=" * 60)

print(customer_output)

Customer Features Saved Successfully
c:\N_VsCode\Zidio Project\PROJECT\RetailPulse-AI-Customer-Analytics\data\processed\customer_features.csv


In [11]:
# ============================================================
# Dataset Summary
# ============================================================

print("Customer Feature Dataset Summary")
print("-" * 50)

print("Rows    :", customer_features.shape[0])
print("Columns :", customer_features.shape[1])

print("\nMissing Values")
print(customer_features.isnull().sum())

print("\nDescriptive Statistics")
display(customer_features.describe())

Customer Feature Dataset Summary
--------------------------------------------------
Rows    : 5878
Columns : 14

Missing Values
CustomerID              0
Recency                 0
Frequency               0
Monetary                0
TotalQuantity           0
AverageOrderValue       0
UniqueProducts          0
UniqueCountries         0
FirstPurchase           0
LastPurchase            0
CustomerLifetimeDays    0
ActiveMonths            0
RevenuePerMonth         0
AvgQuantityPerOrder     0
dtype: int64

Descriptive Statistics


,CustomerID,Recency,Frequency,Monetary,TotalQuantity,AverageOrderValue,UniqueProducts,UniqueCountries,FirstPurchase,LastPurchase,CustomerLifetimeDays,ActiveMonths,RevenuePerMonth,AvgQuantityPerOrder
count,5878.000000,5878.000000,5878.000000,5878.000000,5878.000000,5878.000000,5878.000000,5878.000000,5878,5878,5878.000000,5878.000000,5878.000000,5878.000000
mean,15315.313542,201.331916,6.289384,2955.904095,1788.695475,48.301822,81.989112,1.002212,2010-08-22 06:57:33.297039616,2011-05-22 16:19:59.469207296,273.022457,9.390796,392.319862,247.556827
min,12346.000000,1.000000,1.000000,2.950000,1.000000,2.135778,1.000000,1.000000,2009-12-01 07:45:00,2009-12-01 09:55:00,0.000000,0.100000,1.640000,1.000000
25%,13833.250000,26.000000,1.000000,342.280000,187.000000,11.564180,19.000000,1.000000,2010-02-09 14:01:15,2010-11-25 10:24:45,0.000000,1.000000,96.385000,91.000000
50%,15314.500000,96.000000,3.000000,867.740000,480.000000,17.367639,45.000000,1.000000,2010-06-27 13:31:30,2011-09-05 11:59:00,220.500000,7.350000,180.925000,153.585000
75%,16797.750000,380.000000,7.000000,2248.305000,1350.000000,24.181900,103.000000,1.000000,2011-01-30 14:30:15,2011-11-14 11:31:15,511.000000,17.000000,350.725000,256.925000
max,18287.000000,739.000000,398.000000,580987.040000,367193.000000,56157.500000,2550.000000,2.000000,2011-12-09 12:16:00,2011-12-09 12:50:00,738.000000,24.600000,39916.500000,87167.000000
std,1715.572666,209.338707,13.009406,14440.852688,8876.297196,780.176769,116.484552,0.046980,NaN,NaN,258.807591,8.328314,1253.827100,1424.480949
